# Matrix pair multiplication with TaskVine

We have N matrices stored as CSV files in a local folder.
We multiply every pair (A, B) and compute the Frobenius norm of the result.
We report the top 5 strongest pairs.

**Scale knobs**
- More files → more tasks (grows as N×(N-1)/2)
- Larger matrices → heavier tasks (matrix multiply is O(n³))

**What this pattern maps to in practice**
- *Network reachability* — if entries are 0/1, A×B counts 2-hop paths between nodes
- *Transformation composition* — if matrices are coordinate transforms, A×B is the composed operation
- *Markov chain mixing* — if rows are probability distributions, A×B gives 2-step transition probabilities

In [1]:
#task function (runs on the worker)

def multiply_and_norm(path_a, path_b):
    import csv
    import numpy as np

    def load(path):
        with open(path, newline="") as f:
            return np.array([[float(v) for v in row]
                             for row in csv.reader(f)])

    A = load(path_a)
    B = load(path_b)
    C = A @ B

    return {
        "name_a": path_a.split("/")[-1].replace(".csv", ""),
        "name_b": path_b.split("/")[-1].replace(".csv", ""),
        "norm":   round(float(np.linalg.norm(C, "fro")), 4),
    }

In [2]:
# manager setup

import os
import glob
import itertools
import ndcctools.taskvine as vine

manager_name = os.environ.get("VINE_MANAGER_NAME")
ports_str    = os.environ.get("VINE_MANAGER_PORTS", "9123,9150")
ports        = [int(p.strip()) for p in ports_str.split(",")]

m = vine.Manager(ports, name=manager_name)
print(f"[manager] Listening on port {m.port}")

[manager] Listening on port 9123


In [3]:
# discover files and declare them ──────────────────────────────────

matrix_paths = sorted(glob.glob("data/matrices/*.csv"))
n            = len(matrix_paths)
num_pairs    = n * (n - 1) // 2

print(f"[manager] Found {n} matrix files")
print(f"[manager] Submitting {num_pairs} tasks  ({n}×{n-1}/2 = {num_pairs} pairs)\n")

# Declare each file once — TaskVine stages it to the worker on demand
declared = {path: m.declare_file(path) for path in matrix_paths}

[manager] Found 10 matrix files
[manager] Submitting 45 tasks  (10×9/2 = 45 pairs)



In [4]:
# ── cell 5 — submit one task per pair ─────────────────────────────────────────

task_map = {}   # task_id → (path_a, path_b)

for path_a, path_b in itertools.combinations(matrix_paths, 2):
    task = vine.PythonTask(multiply_and_norm, path_a, path_b)
    task.add_input(declared[path_a], path_a)
    task.add_input(declared[path_b], path_b)
    tid  = m.submit(task)
    task_map[tid] = (path_a, path_b)

print(f"[manager] All {num_pairs} tasks submitted. Waiting for results...\n")

[manager] All 45 tasks submitted. Waiting for results...



In [5]:
# ── cell 6 — collect results ──────────────────────────────────────────────────

results = []

while not m.empty():
    done = m.wait(5)
    if not done:
        continue
    if done.successful():
        r = done.output
        results.append(r)
        print(f"  {r['name_a']:12s} × {r['name_b']:12s}  "
              f"norm={r['norm']:>12.4f}  "
              f"worker={done.addrport}")
    else:
        path_a, path_b = task_map[done.id]
        print(f"  FAILED: {os.path.basename(path_a)} × {os.path.basename(path_b)} "
              f"— {done.result}")

print(f"\n[manager] All {len(results)} tasks complete.\n")

  matrix_dense_00 × matrix_dense_01  norm=  93676.2841  worker=10.32.93.214:59554
  matrix_dense_00 × matrix_dense_02  norm=  94174.7472  worker=10.32.93.214:59554
  matrix_dense_00 × matrix_dense_03  norm=  94991.5094  worker=10.32.93.214:59554


  matrix_dense_00 × matrix_dense_04  norm=  94334.6568  worker=10.32.93.214:59554
  matrix_dense_00 × matrix_dense_05  norm=  94007.2716  worker=10.32.93.214:59554
  matrix_dense_00 × matrix_dense_06  norm=  93540.5166  worker=10.32.93.214:59554


  matrix_dense_00 × matrix_dense_07  norm=  94251.0811  worker=10.32.93.214:59554
  matrix_dense_00 × matrix_dense_08  norm=  93705.7314  worker=10.32.93.214:59554
  matrix_dense_00 × matrix_dense_09  norm=  94682.8783  worker=10.32.93.214:59554


  matrix_dense_01 × matrix_dense_02  norm=  94698.5984  worker=10.32.93.214:59554
  matrix_dense_01 × matrix_dense_03  norm=  95133.5592  worker=10.32.93.214:59554
  matrix_dense_01 × matrix_dense_04  norm=  94174.0070  worker=10.32.93.214:59554


  matrix_dense_01 × matrix_dense_05  norm=  94431.1448  worker=10.32.93.214:59554
  matrix_dense_01 × matrix_dense_06  norm=  94333.4760  worker=10.32.93.214:59554
  matrix_dense_01 × matrix_dense_07  norm=  93902.2395  worker=10.32.93.214:59554


...

  matrix_dense_04 × matrix_dense_05  norm=  94870.9847  worker=10.32.93.214:59554
  matrix_dense_04 × matrix_dense_06  norm=  94565.8419  worker=10.32.93.214:59554
  matrix_dense_04 × matrix_dense_07  norm=  93343.9853  worker=10.32.93.214:59554


  matrix_dense_04 × matrix_dense_08  norm=  94054.8969  worker=10.32.93.214:59554
  matrix_dense_04 × matrix_dense_09  norm=  94184.3646  worker=10.32.93.214:59554
  matrix_dense_05 × matrix_dense_06  norm=  93819.6816  worker=10.32.93.214:59554


  matrix_dense_05 × matrix_dense_07  norm=  93890.3114  worker=10.32.93.214:59554
  matrix_dense_05 × matrix_dense_08  norm=  94519.8424  worker=10.32.93.214:59554
  matrix_dense_05 × matrix_dense_09  norm=  94278.3253  worker=10.32.93.214:59554


  matrix_dense_06 × matrix_dense_07  norm=  93743.9096  worker=10.32.93.214:59554
  matrix_dense_06 × matrix_dense_08  norm=  94003.8394  worker=10.32.93.214:59554
  matrix_dense_06 × matrix_dense_09  norm=  93793.2818  worker=10.32.93.214:59554


  matrix_dense_07 × matrix_dense_08  norm=  93960.7261  worker=10.32.93.214:59554
  matrix_dense_07 × matrix_dense_09  norm=  94459.1442  worker=10.32.93.214:59554
  matrix_dense_08 × matrix_dense_09  norm=  93860.2717  worker=10.32.93.214:59554

[manager] All 45 tasks complete.



In [6]:
# ── cell 7 — top 5 results ────────────────────────────────────────────────────

top5 = sorted(results, key=lambda r: r["norm"], reverse=True)[:5]

print("Top 5 pairs by Frobenius norm")
print("─" * 48)
for rank, r in enumerate(top5, 1):
    print(f"  {rank}.  {r['name_a']:12s} × {r['name_b']:12s}  norm = {r['norm']:,.4f}")
print("─" * 48)

Top 5 pairs by Frobenius norm
────────────────────────────────────────────────
  1.  matrix_dense_01 × matrix_dense_03  norm = 95,133.5592
  2.  matrix_dense_00 × matrix_dense_03  norm = 94,991.5094
  3.  matrix_dense_04 × matrix_dense_05  norm = 94,870.9847
  4.  matrix_dense_01 × matrix_dense_02  norm = 94,698.5984
  5.  matrix_dense_00 × matrix_dense_09  norm = 94,682.8783
────────────────────────────────────────────────
